# 🌿 PlantGuard AI — Model Training

Uses **Colab's pre-installed TensorFlow** — no version pinning, no conflicts.

### Before you start
1. **Runtime → Change runtime type → T4 GPU** ✅
2. Get `kaggle.json` from [kaggle.com/settings](https://www.kaggle.com/settings) → API → Create New Token
3. Run cells **one by one in order**

## Step 1 — Install packages & fix ml_dtypes (restart required after)

In [ ]:
!pip install -q kaggle 'ml_dtypes>=0.5' -q
print('✅ Done — click Runtime → Restart session, then continue from Step 2')

## Step 2 — Verify environment

In [ ]:
import tensorflow as tf
import keras
import numpy as np
import sys

print(f'Python     : {sys.version}')
print(f'TensorFlow : {tf.__version__}')
print(f'Keras      : {keras.__version__}')
print(f'NumPy      : {np.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

assert tf.config.list_physical_devices('GPU'), '❌ No GPU — go to Runtime → Change runtime type → T4 GPU'
print('\n✅ Environment ready')

## Step 3 — Upload kaggle.json

In [ ]:
import os
from google.colab import files

print('👆 Click "Choose Files" and upload your kaggle.json')
files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ kaggle.json configured')

## Step 4 — Download PlantVillage dataset from Kaggle

In [ ]:
import os

print('📥 Downloading dataset (~1.5 GB)...')
ret = os.system('kaggle datasets download -d vipoooool/new-plant-diseases-dataset')
assert ret == 0, '❌ Download failed — check your kaggle.json is valid'

print('📦 Extracting...')
os.system('unzip -q new-plant-diseases-dataset.zip -d plantdata')
print('✅ Dataset ready')

## Step 5 — Locate train / validation directories

In [ ]:
import os

TRAIN_DIR = VAL_DIR = None
for root, dirs, _ in os.walk('plantdata'):
    if 'train' in dirs and TRAIN_DIR is None:
        TRAIN_DIR = os.path.join(root, 'train')
    if 'valid' in dirs and VAL_DIR is None:
        VAL_DIR = os.path.join(root, 'valid')

assert TRAIN_DIR and os.path.isdir(TRAIN_DIR), '❌ train folder not found'
assert VAL_DIR   and os.path.isdir(VAL_DIR),   '❌ valid folder not found'

print(f'Train : {TRAIN_DIR}  ({len(os.listdir(TRAIN_DIR))} classes)')
print(f'Valid : {VAL_DIR}   ({len(os.listdir(VAL_DIR))} classes)')
print('\nAll available classes:')
for c in sorted(os.listdir(TRAIN_DIR)):
    print(' ', c)

## Step 6 — Filter exactly your 15 classes
Order matches `CLASS_LABELS` in `backend/model.py` exactly.

In [ ]:
import os, shutil, re

YOUR_CLASSES = [
    'Pepper__bell___Bacterial_spot',
    'Pepper__bell___healthy',
    'Potato___Early_blight',
    'Potato___Late_blight',
    'Potato___healthy',
    'Tomato___Bacterial_spot',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy',
]

def normalize(s):
    return re.sub(r'[^a-z0-9]', '', s.lower())

def find_match(cls, folder_list):
    if cls in folder_list: return cls
    nc = normalize(cls)
    for f in folder_list:
        if normalize(f) == nc: return f
    for f in folder_list:
        if nc in normalize(f) or normalize(f) in nc: return f
    return None

FILTERED_TRAIN = 'filtered/train'
FILTERED_VAL   = 'filtered/valid'
os.makedirs(FILTERED_TRAIN, exist_ok=True)
os.makedirs(FILTERED_VAL,   exist_ok=True)

avail_train = os.listdir(TRAIN_DIR)
avail_val   = os.listdir(VAL_DIR)
missing = []

for cls in YOUR_CLASSES:
    mt = find_match(cls, avail_train)
    mv = find_match(cls, avail_val)
    if mt:
        shutil.copytree(os.path.join(TRAIN_DIR, mt), os.path.join(FILTERED_TRAIN, cls), dirs_exist_ok=True)
        shutil.copytree(os.path.join(VAL_DIR,   mv), os.path.join(FILTERED_VAL,   cls), dirs_exist_ok=True)
        print(f'✅ {cls}')
    else:
        missing.append(cls)
        print(f'❌ NOT FOUND: {cls}')

print(f'\n{15 - len(missing)}/15 classes ready')
if missing:
    raise RuntimeError(f'❌ Missing: {missing}')

## Step 7 — Build data pipelines

In [ ]:
import tensorflow as tf

IMG_SIZE = 224
BATCH    = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    FILTERED_TRAIN,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    label_mode='categorical',
    shuffle=True,
    class_names=YOUR_CLASSES
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    FILTERED_VAL,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    label_mode='categorical',
    shuffle=False,
    class_names=YOUR_CLASSES
)

norm     = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (norm(x), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.map(lambda x, y:   (norm(x), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

print(f'✅ Train batches : {len(train_ds)}')
print(f'✅ Val batches   : {len(val_ds)}')

## Step 8 — Build MobileNetV2 model

In [ ]:
from keras import layers, Model

NUM_CLASSES = 15

base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base.trainable = False

augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name='augmentation')

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = augment(inputs)
x       = base(x, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## Step 9 — Phase 1: Train top layers (10 epochs)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.3, patience=2, monitor='val_loss', verbose=1)
]

print('🚀 Phase 1: Training top layers...')
history1 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)
print(f'✅ Phase 1 done — best val_accuracy: {max(history1.history["val_accuracy"]):.4f}')

## Step 10 — Phase 2: Fine-tune top 40 layers (10 more epochs)

In [ ]:
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('🚀 Phase 2: Fine-tuning...')
history2 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)
print(f'✅ Phase 2 done — best val_accuracy: {max(history2.history["val_accuracy"]):.4f}')

## Step 11 — Evaluate & plot

In [ ]:
import matplotlib.pyplot as plt

loss, acc = model.evaluate(val_ds, verbose=1)
print(f'\n🎯 Final Validation Accuracy : {acc*100:.2f}%')
print(f'📉 Final Validation Loss     : {loss:.4f}')

acc_all = history1.history['accuracy']     + history2.history['accuracy']
val_all = history1.history['val_accuracy'] + history2.history['val_accuracy']
split   = len(history1.history['accuracy'])

plt.figure(figsize=(10, 4))
plt.plot(acc_all, label='Train', color='#2d6a4f')
plt.plot(val_all, label='Val',   color='#52b788')
plt.axvline(x=split, color='gray', linestyle='--', label='Fine-tune start')
plt.title('Training Accuracy'); plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.tight_layout(); plt.show()

## Step 12 — Save & verify model

In [ ]:
import numpy as np

model.save('model.keras')
model.save('model.h5')
print('✅ model.keras saved')
print('✅ model.h5 saved')

loaded = tf.keras.models.load_model('model.keras')
preds  = loaded.predict(np.zeros((1, 224, 224, 3), dtype='float32'), verbose=0)
assert preds.shape == (1, 15), f'❌ Expected (1,15), got {preds.shape}'
print(f'✅ Sanity check passed — output shape: {preds.shape}')

## Step 13 — Download model files

Place downloaded files in your local project:
```
backend/model/model.keras   ← primary
backend/model/model.h5      ← fallback
```

In [ ]:
from google.colab import files
files.download('model.keras')
files.download('model.h5')
print('✅ Place both files in backend/model/')